# Step 3: Attention

## 3.1 Ne Değişiyor?

Önceki model, sonraki tokenı yalnızca mevcut tokena göre
tahmin ediyordu. Yeni model, konum bilgisini ve önceki
tokenlardan gelen bilgileri de kullanacak.

Bu bölümde konum embeddingleri, RMSNorm, tek başlı attention,
residual bağlantılar, çıktı katmanı ve KV cache eklenecek.
Veri kümesi, tokenizer, Value sınıfı ve SGD yöntemi korunacak.

In [1]:
import random
import math
with open("../data/raw_names.txt", encoding="utf-8") as f:
    docs = [
        line.strip()
        .replace("İ", "i")
        .replace("I", "ı")
        .lower()
        .replace(" ", "")
        for line in f
        if line.strip()
    ]

random.shuffle(docs)

uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1

print("İsim sayısı:", len(docs))
print("Sözlük boyutu:", vocab_size)

İsim sayısı: 1100
Sözlük boyutu: 30


## 3.2 Konum Embeddingleri

Token embeddingi, hangi tokenın işlendiğini temsil eder.
Konum embeddingi ise tokenın dizideki konumunu temsil eder.

Bu iki vektör eleman eleman toplanır. Böylece aynı token,
farklı konumlarda farklı giriş vektörleriyle temsil edilir.
Her iki embedding tablosu da eğitim sırasında öğrenilir.

In [2]:



class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(
            self.data * other.data,
            (self, other),
            (other.data, self.data)
        )

    def __pow__(self, other):
        return Value(
            self.data ** other,
            (self,),
            (other * self.data ** (other - 1),)
        )

    def log(self):
        return Value(
            math.log(self.data),
            (self,),
            (1 / self.data,)
        )

    def exp(self):
        return Value(
            math.exp(self.data),
            (self,),
            (math.exp(self.data),)
        )

    def relu(self):
        return Value(
            max(0, self.data),
            (self,),
            (float(self.data > 0),)
        )

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

    # 2.5 Backward
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

In [3]:

n_embd = 16

block_size = 16

matrix = lambda nout, nin: [
    [Value(random.gauss(0, 0.08)) for _ in range(nin)]
    for _ in range(nout)
]

state_dict = {
    'wte': matrix(vocab_size, n_embd),
    'wpe': matrix(block_size, n_embd),
    'attn_wq': matrix(n_embd, n_embd),
    'attn_wk': matrix(n_embd, n_embd),
    'attn_wv': matrix(n_embd, n_embd),
    'attn_wo': matrix(n_embd, n_embd),
    'mlp_fc1': matrix(4 * n_embd, n_embd),
    'mlp_fc2': matrix(n_embd, 4 * n_embd),
    'lm_head': matrix(vocab_size, n_embd),
}

In [4]:
token_id = uchars.index("e")
pos_id = 1

tok_emb = state_dict['wte'][token_id]
pos_emb = state_dict['wpe'][pos_id]

x = [t + p for t, p in zip(tok_emb, pos_emb)]

print("Token embeddingi:", [v.data for v in tok_emb[:3]])
print("Konum embeddingi:", [v.data for v in pos_emb[:3]])
print("Birleşik vektör:", [v.data for v in x[:3]])
print("Vektör uzunluğu:", len(x))

Token embeddingi: [0.02623590482434665, -0.018443518781795322, -0.027658457542015423]
Konum embeddingi: [0.031088956860132348, 0.10867712598240047, 0.042849083233824005]
Birleşik vektör: [0.057324861684479, 0.09023360720060514, 0.015190625691808582]
Vektör uzunluğu: 16


## 3.3 RMSNorm

RMSNorm, vektördeki değerlerin karelerinin ortalamasını hesaplar.
Bu ortalamanın kareköküne göre bütün değerleri aynı oranda
ölçeklendirir. Sonuçta vektörün RMS değeri yaklaşık 1 olur.

Bu işlem, katmanlara giren değerlerin ölçeğini düzenlemeye
yardımcı olur. Sıfıra bölünmeyi önlemek için küçük bir epsilon
değeri eklenir. LayerNorm'dan farklı olarak ortalama çıkarılmaz.

In [5]:
def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

In [6]:
x_norm = rmsnorm(x)

rms_before = (sum(v.data ** 2 for v in x) / len(x)) ** 0.5
rms_after = (
    sum(v.data ** 2 for v in x_norm) / len(x_norm)
) ** 0.5

print("Önceki RMS:", rms_before)
print("Sonraki RMS:", rms_after)
print("İlk 3 normalize değer:", [v.data for v in x_norm[:3]])
print("Vektör uzunluğu:", len(x_norm))

Önceki RMS: 0.13894118007860481
Sonraki RMS: 0.9997410956475667
İlk 3 normalize değer: [0.4124768480868205, 0.6492693186852843, 0.10930303574583894]
Vektör uzunluğu: 16


## 3.4 Attention için Q, K ve V

Normalize edilmiş token ve konum vektöründen üç farklı
ağırlık matrisiyle Q, K ve V vektörleri oluşturulur.

Q, mevcut tokenın aradığı bilgiyi temsil eder.
K, tokenların bu aramayla karşılaştırılacak temsilidir.
V ise attention sonucunda aktarılacak bilgileri taşır.

Her üç vektör de 16 elemanlıdır. Bu aşamada yalnızca
vektörler oluşturulur; karşılaştırma sonraki aşamadadır.

In [7]:
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

In [8]:
token_id = uchars.index("e")
pos_id = 1

tok_emb = state_dict['wte'][token_id]
pos_emb = state_dict['wpe'][pos_id]
x = [t + p for t, p in zip(tok_emb, pos_emb)]

x_residual = x
x = rmsnorm(x)

q = linear(x, state_dict['attn_wq'])
k = linear(x, state_dict['attn_wk'])
v = linear(x, state_dict['attn_wv'])

print("Q uzunluğu:", len(q))
print("K uzunluğu:", len(k))
print("V uzunluğu:", len(v))

print("Q ilk 3 değer:", [a.data for a in q[:3]])
print("K ilk 3 değer:", [a.data for a in k[:3]])
print("V ilk 3 değer:", [a.data for a in v[:3]])

Q uzunluğu: 16
K uzunluğu: 16
V uzunluğu: 16
Q ilk 3 değer: [0.2894763184520399, 0.328226904512044, -0.3917762507269244]
K ilk 3 değer: [0.15853389147456504, -0.9251066319256106, 0.7047221077894867]
V ilk 3 değer: [0.5965698267407534, 0.38714924718056454, 0.03376866665483366]


## 3.5 Attention Skorları ve Ağırlıkları

Mevcut tokenın Q vektörü, o ana kadar işlenen tokenların
K vektörleriyle noktasal çarpım kullanılarak karşılaştırılır.
Skorlar embedding boyutunun kareköküne bölünür.

Softmax ile skorlar toplamı 1 olan attention ağırlıklarına
dönüştürülür. Mevcut token da hesaba katılır; gelecekteki
tokenlar kullanılmaz.

In [9]:
def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

In [10]:
tokens = [BOS] + [uchars.index(ch) for ch in "efe"]
keys = []
values = []

for pos_id, token_id in enumerate(tokens):
    tok_emb = state_dict['wte'][token_id]
    pos_emb = state_dict['wpe'][pos_id]
    x = [t + p for t, p in zip(tok_emb, pos_emb)]

    x_residual = x
    x = rmsnorm(x)

    q = linear(x, state_dict['attn_wq'])
    k = linear(x, state_dict['attn_wk'])
    v = linear(x, state_dict['attn_wv'])

    keys.append(k)
    values.append(v)

    attn_logits = [
        sum(q[j] * keys[t][j] for j in range(n_embd))
        / n_embd ** 0.5
        for t in range(len(keys))
    ]
    attn_weights = softmax(attn_logits)

    print("Konum:", pos_id, "| Ağırlıklar:",
          [round(w.data, 4) for w in attn_weights])

print("Son ağırlıkların toplamı:", sum(w.data for w in attn_weights))

Konum: 0 | Ağırlıklar: [1.0]
Konum: 1 | Ağırlıklar: [0.5164, 0.4836]
Konum: 2 | Ağırlıklar: [0.3712, 0.3279, 0.3009]
Konum: 3 | Ağırlıklar: [0.2271, 0.28, 0.2302, 0.2627]
Son ağırlıkların toplamı: 0.9999999999999999


## 3.6 Attention Çıktısı ve Residual Bağlantılar

Attention ağırlıklarıyla V vektörleri birleştirilir.
Sonuç, attn_wo matrisinden geçirilip başlangıç girdisine
eklenir. Bu işleme residual bağlantı denir.

MLP bloğunda da normalizasyon, hesaplama ve residual toplama
uygulanır. Son olarak lm_head, 30 token için tahmin skorları
üretir. Bütün işlemler gpt() fonksiyonunda birleştirilir.

In [11]:
def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id]
    pos_emb = state_dict['wpe'][pos_id]
    x = [t + p for t, p in zip(tok_emb, pos_emb)]

    # 1) Attention block
    x_residual = x
    x = rmsnorm(x)
    q = linear(x, state_dict['attn_wq'])
    k = linear(x, state_dict['attn_wk'])
    v = linear(x, state_dict['attn_wv'])

    keys.append(k)
    values.append(v)
    attn_logits = [
        sum(q[j] * keys[t][j] for j in range(n_embd))
        / n_embd ** 0.5
        for t in range(len(keys))
    ]
    attn_weights = softmax(attn_logits)

    x_attn = [
        sum(attn_weights[t] * values[t][j] for t in range(len(values)))
        for j in range(n_embd)
    ]
    x = linear(x_attn, state_dict['attn_wo'])
    x = [a + b for a, b in zip(x, x_residual)]

    # 2) MLP block
    x_residual = x
    x = rmsnorm(x)
    x = linear(x, state_dict['mlp_fc1'])
    x = [xi.relu() for xi in x]
    x = linear(x, state_dict['mlp_fc2'])
    x = [a + b for a, b in zip(x, x_residual)]

    logits = linear(x, state_dict['lm_head'])
    return logits

In [12]:
tokens = [BOS] + [uchars.index(ch) for ch in "efe"]
keys = []
values = []

for pos_id, token_id in enumerate(tokens):
    logits = gpt(token_id, pos_id, keys, values)

probs = softmax(logits)
print("Logit sayısı:", len(logits))
print("Saklanan konum sayısı:", len(keys))
print("Olasılık toplamı:", sum(p.data for p in probs))

Logit sayısı: 30
Saklanan konum sayısı: 4
Olasılık toplamı: 1.0000000000000002


## 3.7 KV Cache

KV cache, işlenen tokenların K ve V vektörlerini saklar.
Her yeni tokenın Q vektörü, saklanan K vektörleriyle
karşılaştırılır. Böylece geçmiş K ve V'ler yeniden hesaplanmaz.

Her yeni isim için cache boş başlatılır. Eğitim sırasında
saklanan Value nesneleri hesaplama grafiğinin parçasıdır;
gradyanlar bu nesneler üzerinden de geriye yayılır.

In [13]:
tokens = [BOS] + [uchars.index(ch) for ch in "efe"]
keys, values = [], []

for pos_id, token_id in enumerate(tokens):
    logits = gpt(token_id, pos_id, keys, values)

    token_text = "BOS" if token_id == BOS else uchars[token_id]

    print(
        f"Konum: {pos_id} | Token: {token_text} | "
        f"K sayısı: {len(keys)} | V sayısı: {len(values)}"
    )

Konum: 0 | Token: BOS | K sayısı: 1 | V sayısı: 1
Konum: 1 | Token: e | K sayısı: 2 | V sayısı: 2
Konum: 2 | Token: f | K sayısı: 3 | V sayısı: 3
Konum: 3 | Token: e | K sayısı: 4 | V sayısı: 4


## 3.8 Eğitim ve Sonuçlar

Model, 1000 adım boyunca SGD ile eğitilir. Learning rate 0.1
olarak başlar ve zamanla azalır. Her isim için KV cache
sıfırlanır; en fazla block_size kadar geçiş işlenir.

Ortalama loss üzerinden backward() çalıştırılır, ağırlıklar
güncellenir ve gradyanlar sıfırlanır. Eğitimden sonra modelin
olasılıkları kullanılarak yeni isimler üretilir.

In [14]:
params = [p for mat in state_dict.values() for row in mat for p in row]

for p in params:
    p.grad = 0

print("Parametre sayısı:", len(params))

Parametre sayısı: 4288


In [15]:
num_steps = 1000
learning_rate = 0.1

for step in range(num_steps):
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)
    keys, values = [], []
    losses = []

    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)

    loss = (1 / n) * sum(losses)
    loss.backward()

    lr_t = learning_rate * (1 - step / num_steps)
    for p in params:
        p.data -= lr_t * p.grad
        p.grad = 0

    if step % 100 == 0 or step == num_steps - 1:
        print(
            f"Adım: {step + 1:4d} | Loss: {loss.data:.4f} | "
            f"Learning rate: {lr_t:.4f}"
        )

Adım:    1 | Loss: 3.4081 | Learning rate: 0.1000
Adım:  101 | Loss: 2.5616 | Learning rate: 0.0900
Adım:  201 | Loss: 2.6266 | Learning rate: 0.0800
Adım:  301 | Loss: 2.6521 | Learning rate: 0.0700
Adım:  401 | Loss: 1.9845 | Learning rate: 0.0600
Adım:  501 | Loss: 2.6174 | Learning rate: 0.0500
Adım:  601 | Loss: 2.3806 | Learning rate: 0.0400
Adım:  701 | Loss: 1.9358 | Learning rate: 0.0300
Adım:  801 | Loss: 1.6839 | Learning rate: 0.0200
Adım:  901 | Loss: 2.4566 | Learning rate: 0.0100
Adım: 1000 | Loss: 2.1767 | Learning rate: 0.0001


In [16]:
temperature = 0.5

for sample_idx in range(20):
    keys, values = [], []
    token_id = BOS
    sample = []

    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits])
        token_id = random.choices(
            range(vocab_size),
            weights=[p.data for p in probs]
        )[0]

        if token_id == BOS:
            break
        sample.append(uchars[token_id])

    print(f"sample {sample_idx + 1:2d}: {''.join(sample)}")

sample  1: sevde
sample  2: enti
sample  3: berma
sample  4: saran
sample  5: başün
sample  6: merdar
sample  7: amra
sample  8: makan
sample  9: hamen
sample 10: aydal
sample 11: nara
sample 12: nesiy
sample 13: neyan
sample 14: helin
sample 15: serin
sample 16: beran
sample 17: emime
sample 18: saliy
sample 19: sedit
sample 20: satil
